In [1]:
import os
import re
import glob
import math
from datetime import datetime, timedelta
from collections import defaultdict

import numpy as np
import rasterio
from rasterio.windows import Window

# =========================
# CONFIG
# =========================
out_dir = r"D:\Thirst_Wave_Outputs\04_Monthly_Thirst_Wave"
INPUT_DIR = r"D:\Thirst_Wave_New_40_Years_Data"   # <- your folder with daily ET0 GeoTIFFs
OUT_MONTHLY_DIR = os.path.join(out_dir, "monthly_by_year")       # per-year monthly sums
OUT_CLIM_DIR    = os.path.join(out_dir, "monthly_climatology")   # 1980-2024 mean of monthly sums
YEARS = set(range(1980, 2025))  # inclusive 1980..2024

# Optional: accepted file extensions
EXTS = (".tif", ".tiff")

os.makedirs(OUT_MONTHLY_DIR, exist_ok=True)
os.makedirs(OUT_CLIM_DIR, exist_ok=True)

# =========================
# HELPERS
# =========================

# Try several filename patterns:
#  - 1980001.tif            -> YYYY JJJ
#  - 1980_001.tif           -> YYYY _ JJJ
#  - 1980-001_ET0.tif       -> YYYY - JJJ
#  - 19800101.tif           -> YYYY MM DD
#  - 1980-01-01_ET0.tif     -> YYYY-MM-DD
DOY_PATTERNS = [
    re.compile(r"(?P<year>\d{4})[_-]?(?P<doy>\d{3})\D"),      # 1980001, 1980_001, 1980-001
    re.compile(r"(?P<year>\d{4})[_-](?P<doy>\d{3})(?:\.tif|\.tiff)$", re.IGNORECASE),
]
YMD_PATTERNS = [
    re.compile(r"(?P<year>\d{4})(?P<month>\d{2})(?P<day>\d{2})\D"), # 19800101
    re.compile(r"(?P<year>\d{4})[-_](?P<month>\d{2})[-_](?P<day>\d{2})\D"), # 1980-01-01 or 1980_01_01
]

def parse_date_from_name(name: str):
    """
    Try to parse date from a filename using common YYYYJJJ or YYYYMMDD patterns.
    Returns a datetime.date or None.
    """
    base = os.path.basename(name) + "_"  # add trailing non-digit to help regex end
    # Try DOY patterns
    for pat in DOY_PATTERNS:
        m = pat.search(base)
        if m:
            year = int(m.group("year"))
            doy = int(m.group("doy"))
            try:
                dt = datetime(year, 1, 1) + timedelta(days=doy - 1)
                return dt.date()
            except Exception:
                pass
    # Try YMD patterns
    for pat in YMD_PATTERNS:
        m = pat.search(base)
        if m:
            year = int(m.group("year"))
            month = int(m.group("month"))
            day = int(m.group("day"))
            try:
                return datetime(year, month, day).date()
            except Exception:
                pass
    return None

def list_daily_files(root):
    paths = []
    for ext in EXTS:
        paths.extend(glob.glob(os.path.join(root, f"**/*{ext}"), recursive=True))
    return sorted(paths)

def init_out_raster_like(template_path, out_path, dtype="float32", nodata=np.float32(np.nan)):
    """
    Create an empty (zeros) GeoTIFF matching the template's size/transform/crs.
    """
    with rasterio.open(template_path) as src:
        profile = src.profile.copy()
        profile.update(
            dtype=dtype,
            count=1,
            nodata=nodata,
            compress="deflate",
            tiled=True,
            bigtiff="IF_SAFER",
            predictor=2
        )
        with rasterio.open(out_path, "w", **profile) as dst:
            # initialize with 0 where valid, NaN will be treated via mask
            zeros = np.zeros((src.block_shapes[0][0], src.block_shapes[0][1]), dtype=dtype)  # not used directly
            # Instead write zeros across full raster efficiently by blocks
            for ji, window in dst.block_windows(1):
                arr = np.zeros((window.height, window.width), dtype=dtype)
                dst.write(arr, 1, window=window)

def add_raster_to_sum(in_path, sum_path, count_path=None):
    """
    Block-wise: sum daily ET0 into sum raster.
    Optionally keep a count raster of valid pixels (for later averaging).
    """
    with rasterio.open(in_path) as src, rasterio.open(sum_path, "r+") as dst_sum:
        # open count raster if provided
        dst_cnt = rasterio.open(count_path, "r+") if count_path else None
        try:
            for ji, window in src.block_windows(1):
                data = src.read(1, window=window, masked=True)  # masked array honors nodata
                # Read current sums
                cur = dst_sum.read(1, window=window)
                # Add only where data is valid
                valid = ~data.mask
                cur[valid] = cur[valid] + data.data[valid].astype(cur.dtype, copy=False)
                dst_sum.write(cur, 1, window=window)

                if dst_cnt is not None:
                    cnt = dst_cnt.read(1, window=window)
                    cnt[valid] = cnt[valid] + 1
                    dst_cnt.write(cnt, 1, window=window)
        finally:
            if dst_cnt is not None:
                dst_cnt.close()

def average_rasters(in_paths, out_path):
    """
    Compute per-pixel average over a list of rasters with nodata handling, block-wise.
    Uses a temporary count raster internally for robustness.
    Assumes all rasters are same grid/shape.
    """
    if not in_paths:
        return

    template = in_paths[0]
    # Prepare sum and count outputs
    sum_path = out_path + ".tmp_sum.tif"
    cnt_path = out_path + ".tmp_cnt.tif"

    # Create sum and count rasters
    init_out_raster_like(template, sum_path, dtype="float32", nodata=np.float32(np.nan))
    init_out_raster_like(template, cnt_path, dtype="uint16", nodata=0)

    # Accumulate
    for fp in in_paths:
        add_raster_to_sum(fp, sum_path, count_path=cnt_path)

    # Create final average
    with rasterio.open(template) as src_t, \
         rasterio.open(sum_path) as src_sum, \
         rasterio.open(cnt_path) as src_cnt:

        profile = src_t.profile.copy()
        profile.update(dtype="float32", nodata=np.float32(np.nan), compress="deflate", tiled=True, predictor=2, bigtiff="IF_SAFER")

        with rasterio.open(out_path, "w", **profile) as dst_avg:
            for ji, window in src_sum.block_windows(1):
                s = src_sum.read(1, window=window)
                c = src_cnt.read(1, window=window).astype("float32")
                with np.errstate(invalid="ignore", divide="ignore"):
                    avg = np.where(c > 0, s / c, np.float32(np.nan))
                dst_avg.write(avg.astype("float32"), 1, window=window)

    # Clean up temps
    try:
        os.remove(sum_path)
    except Exception:
        pass
    try:
        os.remove(cnt_path)
    except Exception:
        pass

# =========================
# MAIN: build (year, month) sums, then climatology
# =========================

# 1) index daily files by date
daily_files = list_daily_files(INPUT_DIR)
date_to_file = {}
for fp in daily_files:
    dt = parse_date_from_name(fp)
    if dt is None:
        continue
    if dt.year in YEARS:
        date_to_file[dt] = fp

if not date_to_file:
    raise RuntimeError("No daily files with recognizable dates found for 1980–2024.")

# 2) group by (year, month)
files_by_ym = defaultdict(list)
for dt, fp in date_to_file.items():
    files_by_ym[(dt.year, dt.month)].append(fp)

# 3) create monthly sums per year
monthly_outputs = defaultdict(str)  # (year, month) -> path
example_template = next(iter(date_to_file.values()))  # for grid template

for (yy, mm), flist in sorted(files_by_ym.items()):
    # Output path for monthly sum
    out_name = f"ET0_sum_{yy}_{mm:02d}.tif"
    out_path = os.path.join(OUT_MONTHLY_DIR, out_name)
    monthly_outputs[(yy, mm)] = out_path

    if os.path.exists(out_path):
        # skip if already done
        continue

    # Init accumulator (sum) raster
    init_out_raster_like(example_template, out_path, dtype="float32", nodata=np.float32(np.nan))

    # Add each daily raster into the monthly sum
    for fp in sorted(flist):
        add_raster_to_sum(fp, out_path)

    print(f"[OK] Monthly sum written: {out_path}  (from {len(flist)} daily files)")

# 4) Build monthly climatology (1980–2024 mean of monthly sums)
# For each calendar month, collect all yearly monthly-sum rasters and average
for mm in range(1, 13):
    month_stack = []
    for yy in sorted(YEARS):
        key = (yy, mm)
        if key in monthly_outputs and os.path.exists(monthly_outputs[key]):
            month_stack.append(monthly_outputs[key])

    if not month_stack:
        print(f"[WARN] No monthly sums found for month {mm:02d}")
        continue

    clim_out = os.path.join(OUT_CLIM_DIR, f"ET0_monthly_climatology_1980_2024_{mm:02d}.tif")
    if os.path.exists(clim_out):
        print(f"[SKIP] Climatology exists for month {mm:02d}")
        continue

    average_rasters(month_stack, clim_out)
    print(f"[OK] Climatology written: {clim_out}  (n_years = {len(month_stack)})")

print("\nDONE.")
print(f"Per-year monthly sums → {OUT_MONTHLY_DIR}")
print(f"Monthly climatology (1980–2024 means) → {OUT_CLIM_DIR}")


[OK] Monthly sum written: D:\Thirst_Wave_Outputs\04_Monthly_Thirst_Wave\monthly_by_year\ET0_sum_1980_01.tif  (from 31 daily files)
[OK] Monthly sum written: D:\Thirst_Wave_Outputs\04_Monthly_Thirst_Wave\monthly_by_year\ET0_sum_1980_02.tif  (from 29 daily files)
[OK] Monthly sum written: D:\Thirst_Wave_Outputs\04_Monthly_Thirst_Wave\monthly_by_year\ET0_sum_1980_03.tif  (from 31 daily files)
[OK] Monthly sum written: D:\Thirst_Wave_Outputs\04_Monthly_Thirst_Wave\monthly_by_year\ET0_sum_1980_04.tif  (from 30 daily files)
[OK] Monthly sum written: D:\Thirst_Wave_Outputs\04_Monthly_Thirst_Wave\monthly_by_year\ET0_sum_1980_05.tif  (from 31 daily files)
[OK] Monthly sum written: D:\Thirst_Wave_Outputs\04_Monthly_Thirst_Wave\monthly_by_year\ET0_sum_1980_06.tif  (from 30 daily files)
[OK] Monthly sum written: D:\Thirst_Wave_Outputs\04_Monthly_Thirst_Wave\monthly_by_year\ET0_sum_1980_07.tif  (from 31 daily files)
[OK] Monthly sum written: D:\Thirst_Wave_Outputs\04_Monthly_Thirst_Wave\monthly_by_

In [2]:
import os
import re
import numpy as np
import rasterio

aggr_mon = r"D:\Thirst_Wave_Outputs\04_Monthly_Thirst_Wave\monthly_aggr"
MONTHLY_DIR = r"D:\Thirst_Wave_Outputs\04_Monthly_Thirst_Wave\monthly_climatology"
OUT_DIR = os.path.join(aggr_mon, "seasonal_climatology_rolling")
os.makedirs(OUT_DIR, exist_ok=True)

# Seasons in your requested order (rolling 3-month windows)
SEASONS = [
    ("DJF", [12, 1, 2]),
    ("NDJ", [11, 12, 1]),
    ("OND", [10, 11, 12]),
    ("SON", [9, 10, 11]),
    ("ASO", [8, 9, 10]),
    ("JAS", [7, 8, 9]),
    ("JJA", [6, 7, 8]),
    ("MJJ", [5, 6, 7]),
    ("AMJ", [4, 5, 6]),
    ("MAM", [3, 4, 5]),
    ("FMA", [2, 3, 4]),
    ("JFM", [1, 2, 3]),
]

# --- find monthly files robustly (expects one file per month) ---
tif_paths = [os.path.join(MONTHLY_DIR, f) for f in os.listdir(MONTHLY_DIR) if f.lower().endswith(".tif")]

# Try to map month -> path by detecting 2-digit month tokens in filenames
month_to_path = {}
for p in tif_paths:
    fname = os.path.basename(p)
    # Look for tokens like _MM, -MM, or ...MM.tif (where MM is 01..12)
    for m in range(1, 13):
        mm = f"{m:02d}"
        if re.search(rf"(^|[_\-]){mm}(\D|$)", fname):  # token match
            # Prefer first match per month
            month_to_path.setdefault(m, p)

# Sanity check
missing = [m for m in range(1, 13) if m not in month_to_path]
if missing:
    raise RuntimeError(f"Missing monthly climatology files for months: {missing}. "
                       f"Ensure filenames contain tokens like '_MM', '-MM', or end with 'MM.tif'.")

def average_rasters(in_paths, out_path):
    """Pixelwise mean with NoData handling."""
    with rasterio.open(in_paths[0]) as src0:
        profile = src0.profile.copy()
        profile.update(dtype="float32", nodata=np.float32(np.nan), compress="deflate", tiled=True, predictor=2)
        H, W = src0.height, src0.width
        sum_arr = np.zeros((H, W), dtype="float64")
        cnt_arr = np.zeros((H, W), dtype="int32")

        for fp in in_paths:
            with rasterio.open(fp) as src:
                a = src.read(1, masked=True)
                valid = ~a.mask
                sum_arr[valid] += a.data[valid]
                cnt_arr[valid] += 1

        mean = np.where(cnt_arr > 0, sum_arr / cnt_arr, np.nan).astype("float32")

        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(mean, 1)

# Build each rolling season
for sname, months in SEASONS:
    paths = [month_to_path[m] for m in months]
    out_path = os.path.join(OUT_DIR, f"ET0_{sname}_climatology_1980_2024_mean.tif")
    average_rasters(paths, out_path)
    print(f"[OK] {sname} → {out_path}")


C:\Users\maharjan\AppData\Local\Temp\ipykernel_67860\936117828.py:63: RuntimeWarning: invalid value encountered in divide
  mean = np.where(cnt_arr > 0, sum_arr / cnt_arr, np.nan).astype("float32")


[OK] DJF → D:\Thirst_Wave_Outputs\04_Monthly_Thirst_Wave\monthly_aggr\seasonal_climatology_rolling\ET0_DJF_climatology_1980_2024_mean.tif
[OK] NDJ → D:\Thirst_Wave_Outputs\04_Monthly_Thirst_Wave\monthly_aggr\seasonal_climatology_rolling\ET0_NDJ_climatology_1980_2024_mean.tif
[OK] OND → D:\Thirst_Wave_Outputs\04_Monthly_Thirst_Wave\monthly_aggr\seasonal_climatology_rolling\ET0_OND_climatology_1980_2024_mean.tif
[OK] SON → D:\Thirst_Wave_Outputs\04_Monthly_Thirst_Wave\monthly_aggr\seasonal_climatology_rolling\ET0_SON_climatology_1980_2024_mean.tif
[OK] ASO → D:\Thirst_Wave_Outputs\04_Monthly_Thirst_Wave\monthly_aggr\seasonal_climatology_rolling\ET0_ASO_climatology_1980_2024_mean.tif
[OK] JAS → D:\Thirst_Wave_Outputs\04_Monthly_Thirst_Wave\monthly_aggr\seasonal_climatology_rolling\ET0_JAS_climatology_1980_2024_mean.tif
[OK] JJA → D:\Thirst_Wave_Outputs\04_Monthly_Thirst_Wave\monthly_aggr\seasonal_climatology_rolling\ET0_JJA_climatology_1980_2024_mean.tif
[OK] MJJ → D:\Thirst_Wave_Outputs\

In [3]:
import os, re, csv
import numpy as np
import rasterio
from rasterio.enums import ColorInterp

SEASONAL_DIR = r"D:\Thirst_Wave_Outputs\04_Monthly_Thirst_Wave\seasonal_climatology_rolling"

# Your 12 rolling seasons, in this exact order (codes = 1..12)
SEASONS = ["DJF","NDJ","OND","SON","ASO","JAS","JJA","MJJ","AMJ","MAM","FMA","JFM"]

# Find files by token in name (expects filenames like ..._DJF_...tif etc.)
season_to_path = {s: None for s in SEASONS}
for f in os.listdir(SEASONAL_DIR):
    if not f.lower().endswith(".tif"):
        continue
    for s in SEASONS:
        if re.search(rf"(^|[_\-]){s}([_\-]|\.tif$)", f, re.IGNORECASE):
            season_to_path[s] = os.path.join(SEASONAL_DIR, f)

missing = [s for s,p in season_to_path.items() if p is None]
if missing:
    raise RuntimeError(f"Missing seasonal TIFF(s): {missing}")

ordered_paths = [season_to_path[s] for s in SEASONS]

out_idx = os.path.join(SEASONAL_DIR, "season_index.tif")
out_idx_col = os.path.join(SEASONAL_DIR, "season_index_colored.tif")
out_leg = os.path.join(SEASONAL_DIR, "season_legend.csv")

# Make a pleasant 12-class color table (RGB 0-255). 0 is NoData (transparent-ish grey).
COLORS = {
    0:(230,230,230),
    1:(0,76,153),   # DJF
    2:(0,120,180),  # NDJ
    3:(30,144,255), # OND
    4:(0,168,132),  # SON
    5:(60,179,113), # ASO
    6:(102,205,170),# JAS
    7:(255,165,0),  # JJA
    8:(255,140,0),  # MJJ
    9:(255,99,71),  # AMJ
    10:(220,20,60), # MAM
    11:(186,85,211),# FMA
    12:(148,0,211), # JFM
}

with rasterio.open(ordered_paths[0]) as src0:
    profile = src0.profile.copy()
    profile.update(count=1, dtype="uint8", nodata=0, compress="deflate", tiled=True)

# Build index (winner) raster block-wise
with rasterio.open(out_idx, "w", **profile) as dst_idx, \
     rasterio.open(ordered_paths[0]) as src_ref:

    for ji, window in src_ref.block_windows(1):
        # Stack season rasters
        stack = []
        masks = []
        for p in ordered_paths:
            with rasterio.open(p) as src:
                a = src.read(1, window=window, masked=True).astype("float32")
                stack.append(a.data)
                masks.append(a.mask)
        stack = np.stack(stack, axis=0)    # (12,h,w)
        masks = np.stack(masks, axis=0)    # (12,h,w)

        valid_any = ~np.all(masks, axis=0)
        stack_masked = np.where(masks, -np.inf, stack)

        arg = np.argmax(stack_masked, axis=0)       # 0..11
        idx = (arg + 1).astype("uint8")             # 1..12
        idx[~valid_any] = 0                         # NoData code

        dst_idx.write(idx, 1, window=window)

# Write legend CSV (code -> label)
with open(out_leg, "w", newline="") as f:
    w = csv.writer(f); w.writerow(["code","season"])
    w.writerow([0,"NoData"])
    for i,s in enumerate(SEASONS, start=1):
        w.writerow([i,s])

# Create a colored copy with an embedded palette (nice for QGIS/ArcGIS)
with rasterio.open(out_idx) as src:
    prof = src.profile.copy()
    with rasterio.open(out_idx_col, "w", **prof) as dst:
        dst.write(src.read(1), 1)
        # Set a color table on band 1
        cmap = {k: (*v, 255) for k,v in COLORS.items()}  # add full alpha
        dst.write_colormap(1, cmap)
        dst.set_band_description(1, "Season code (1..12); see season_legend.csv")
        dst.colorinterp = (ColorInterp.palette,)  # mark band as palette-indexed

print("[OK] Wrote:")
print(" -", out_idx)
print(" -", out_idx_col, "(with color table)")
print(" -", out_leg)
print("\nCodes (1..12) map to:", SEASONS)


[OK] Wrote:
 - D:\Thirst_Wave_Outputs\04_Monthly_Thirst_Wave\seasonal_climatology_rolling\season_index.tif
 - D:\Thirst_Wave_Outputs\04_Monthly_Thirst_Wave\seasonal_climatology_rolling\season_index_colored.tif (with color table)
 - D:\Thirst_Wave_Outputs\04_Monthly_Thirst_Wave\seasonal_climatology_rolling\season_legend.csv

Codes (1..12) map to: ['DJF', 'NDJ', 'OND', 'SON', 'ASO', 'JAS', 'JJA', 'MJJ', 'AMJ', 'MAM', 'FMA', 'JFM']
